In [0]:
from pyspark.sql import functions as F

# Origens
TABELA_INEP = (
    "workspace.silver.inep_superior_cursos"
)

TABELA_GOLD_INEP = (
    "workspace.gold.fato_inep_educacao_brasil_anual"
)

# Destinos futuros
DIM_CINE = "workspace.gold.dim_cine_brasil"
DIM_CBO = "workspace.gold.dim_cbo_ocupacao"

# Gold do Novo Caged (verificação de cobertura CBO)
TABELA_GOLD_CAGED = "workspace.gold.fato_caged_semestral"

# Período analítico
ANOS_PROCESSAMENTO = [2021, 2022, 2023, 2024]

# Controle de versão da transformação das dimensões
VERSAO_TRANSFORMACAO_DIMENSOES = "1.0.0"

print("Configurações carregadas.")

In [0]:
df_inep = (
    spark.table(TABELA_INEP)
    .filter(
        F.col("ano_censo").isin(ANOS_PROCESSAMENTO)
    )
)

colunas_cine = [
    "codigo_cine_area_geral",
    "nome_cine_area_geral",
    "codigo_cine_area_especifica",
    "nome_cine_area_especifica",
    "codigo_cine_area_detalhada",
    "nome_cine_area_detalhada",
    "codigo_cine_rotulo",
    "nome_cine_rotulo"
]

df_cine_catalogo = (
    df_inep
    .select(*colunas_cine)
    .distinct()
)

print(
    "Rótulos CINE distintos:",
    df_cine_catalogo
    .select("codigo_cine_rotulo")
    .distinct()
    .count()
)

print(
    "Combinações distintas de classificação:",
    df_cine_catalogo.count()
)

display(
    df_cine_catalogo
    .orderBy("codigo_cine_rotulo")
    .limit(30)
)

In [0]:
schemas = [
    "workspace.bronze",
    "workspace.silver",
    "workspace.gold"
]

for schema in schemas:
    print(f"\nSCHEMA: {schema}")

    tabelas = spark.sql(
        f"SHOW TABLES IN {schema}"
    )

    display(
        tabelas.filter(
            F.lower(F.col("tableName")).contains("cbo")
            | F.lower(F.col("tableName")).contains("ocup")
        )
    )

In [0]:
from pyspark.sql import functions as F

df_divergencias_cine = (
    df_cine_catalogo
    .groupBy("codigo_cine_rotulo")
    .agg(
        F.count("*").alias("qtd_combinacoes"),
        F.countDistinct(
            "nome_cine_rotulo"
        ).alias("qtd_nomes")
    )
    .filter(F.col("qtd_combinacoes") > 1)
)

print(
    "Rótulos com múltiplas combinações:",
    df_divergencias_cine.count()
)

display(
    df_cine_catalogo
    .join(
        df_divergencias_cine.select("codigo_cine_rotulo"),
        "codigo_cine_rotulo",
        "inner"
    )
    .orderBy("codigo_cine_rotulo")
)

In [0]:
TABELA_CBO = "workspace.silver.ref_cbo_ocupacao"

df_cbo = spark.table(TABELA_CBO)

print("Schema da referência CBO:")
df_cbo.printSchema()

print("\nQuantidade total de registros:")
print(df_cbo.count())

print("\nAmostra da referência:")
display(df_cbo.limit(20))

In [0]:
codigos_investigar = ["0613E01", "0714E04"]

display(
    df_cine_catalogo
    .filter(
        F.col("codigo_cine_rotulo").isin(codigos_investigar)
    )
    .orderBy("codigo_cine_rotulo")
)

In [0]:
df_evolucao_cine = (
    df_inep
    .filter(
        F.col("codigo_cine_rotulo").isin(
            "0613E01",
            "0714E04"
        )
    )
    .groupBy(
        "ano_censo",
        "codigo_cine_rotulo",
        "nome_cine_rotulo",
        "codigo_cine_area_detalhada",
        "nome_cine_area_detalhada"
    )
    .agg(
        F.count("*").alias("qtd_registros")
    )
    .orderBy(
        "codigo_cine_rotulo",
        "ano_censo",
        "nome_cine_rotulo"
    )
)

display(df_evolucao_cine)

In [0]:
from pyspark.sql import functions as F

# Catálogo CBO tratado: código como STRING com 6 dígitos (zero à esquerda)
# e descrição sem espaços nas bordas. As validações seguem na célula seguinte.
TABELA_CBO = "workspace.silver.ref_cbo_ocupacao"

df_cbo_tratado = (
    spark.table(TABELA_CBO)
    .select(
        F.lpad(F.col("codigo").cast("STRING"), 6, "0").alias("codigo_cbo"),
        F.trim(F.col("descricao")).alias("nome_ocupacao"),
    )
)

print("Catálogo CBO preparado a partir de:", TABELA_CBO)

display(df_cbo_tratado.limit(10))

In [0]:
chaves_duplicadas = (
    df_cbo_tratado
    .groupBy("codigo_cbo")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

codigos_invalidos = (
    df_cbo_tratado
    .filter(
        F.col("codigo_cbo").isNull()
        | ~F.col("codigo_cbo").rlike("^[0-9]{6}$")
    )
    .count()
)

descricoes_vazias = (
    df_cbo_tratado
    .filter(
        F.col("nome_ocupacao").isNull()
        | (F.col("nome_ocupacao") == "")
    )
    .count()
)

print("Registros:", df_cbo_tratado.count())
print("Códigos distintos:", df_cbo_tratado.select("codigo_cbo").distinct().count())
print("Chaves duplicadas:", chaves_duplicadas)
print("Códigos inválidos:", codigos_invalidos)
print("Descrições nulas/vazias:", descricoes_vazias)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

colunas_cine = [
    "codigo_cine_area_geral",
    "nome_cine_area_geral",
    "codigo_cine_area_especifica",
    "nome_cine_area_especifica",
    "codigo_cine_area_detalhada",
    "nome_cine_area_detalhada",
    "codigo_cine_rotulo",
    "nome_cine_rotulo"
]

# Selecionar classificação mais recente por código
janela = (
    Window
    .partitionBy("codigo_cine_rotulo")
    .orderBy(F.col("ano_censo").desc())
)

df_dim_cine = (
    df_inep
    .select("ano_censo", *colunas_cine)
    .dropDuplicates()
    .withColumn(
        "rn",
        F.row_number().over(janela)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Registros da dimensão CINE:", df_dim_cine.count())
print(
    "Rótulos distintos:",
    df_dim_cine
    .select("codigo_cine_rotulo")
    .distinct()
    .count()
)

display(
    df_dim_cine.filter(
        F.col("codigo_cine_rotulo").isin(
            "0613E01", "0714E04"
        )
    )
)

## Preparação e gravação das dimensões na Gold

A partir daqui o notebook sai da fase exploratória e produz as dimensões analíticas, na ordem operacional (pronta para execução em Jobs):

1. **Preparação validada** — `dim_cine_brasil` (grão: um registro por `codigo_cine_rotulo`, nomenclatura da edição mais recente, com detecção de conflitos no mesmo ano) e `dim_cbo_ocupacao` (grão: um registro por código CBO 2002 com 6 dígitos);
2. **Cobertura** — códigos CBO da Gold do Novo Caged cobertos pela dimensão;
3. **Gravação idempotente** — a dimensão só é reescrita quando o conteúdo da origem ou a versão da transformação mudar;
4. **Validação pós-gravação** — releitura das tabelas persistidas.

Fora do escopo desta etapa: `bridge_cine_cbo` e alterações nas Golds existentes.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

colunas_cine = [
    "codigo_cine_area_geral",
    "nome_cine_area_geral",
    "codigo_cine_area_especifica",
    "nome_cine_area_especifica",
    "codigo_cine_area_detalhada",
    "nome_cine_area_detalhada",
    "codigo_cine_rotulo",
    "nome_cine_rotulo",
]

# Base analítica: censos 2021–2024, combinações distintas de classificação
df_combinacoes_cine = (
    spark.table(TABELA_INEP)
    .filter(F.col("ano_censo").isin(ANOS_PROCESSAMENTO))
    .select("ano_censo", *colunas_cine)
    .distinct()
)

# Ano mais recente por rótulo via max() em janela — sem row_number,
# nenhuma linha é escolhida arbitrariamente dentro do ano.
janela_rotulo = Window.partitionBy("codigo_cine_rotulo")

df_ultimo_ano_cine = (
    df_combinacoes_cine
    .withColumn("ano_mais_recente", F.max("ano_censo").over(janela_rotulo))
    .filter(F.col("ano_censo") == F.col("ano_mais_recente"))
    .drop("ano_mais_recente")
)

# Conflitos no mesmo ano: rótulo com mais de uma classificação na edição mais recente
df_conflitos_mesmo_ano = (
    df_ultimo_ano_cine
    .groupBy("codigo_cine_rotulo", "ano_censo")
    .agg(F.count("*").alias("qtd_combinacoes_no_ano"))
    .filter(F.col("qtd_combinacoes_no_ano") > 1)
)

qtd_conflitos = df_conflitos_mesmo_ano.count()

if qtd_conflitos > 0:
    print(f"ATENÇÃO: {qtd_conflitos} rótulos com classificações divergentes no mesmo ano:")
    display(
        df_ultimo_ano_cine
        .join(
            df_conflitos_mesmo_ano.select("codigo_cine_rotulo", "ano_censo"),
            ["codigo_cine_rotulo", "ano_censo"],
            "inner",
        )
        .orderBy("codigo_cine_rotulo")
    )
    raise ValueError(
        "Conflitos de nomenclatura no mesmo ano impedem a seleção "
        "determinística — revise manualmente antes de gravar a dimensão."
    )

# Dimensão final: nomenclatura da edição mais recente de cada rótulo
df_dim_cine = (
    df_ultimo_ano_cine
    .withColumnRenamed("ano_censo", "ano_censo_edicao")
    .select(*colunas_cine, "ano_censo_edicao")
)

resumo_cine = df_dim_cine.agg(
    F.count("*").alias("qtd_registros"),
    F.countDistinct("codigo_cine_rotulo").alias("qtd_rotulos"),
    *[
        F.sum(
            F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)
        ).alias(f"nulos__{c}")
        for c in colunas_cine
    ],
    F.sum(F.when(F.col("ano_censo_edicao").isNull(), 1).otherwise(0)).alias(
        "nulos__ano_censo_edicao"
    ),
).collect()[0]

print("Registros da dimensão CINE:", resumo_cine["qtd_registros"])
print("Rótulos distintos:", resumo_cine["qtd_rotulos"])
for c in colunas_cine + ["ano_censo_edicao"]:
    print(f"  Nulos/vazios em {c}:", resumo_cine[f"nulos__{c}"])

if resumo_cine["qtd_registros"] != 372:
    print(
        f"ALERTA: esperados 372 rótulos, encontrados {resumo_cine['qtd_registros']} "
        "(base pode ter sido atualizada)."
    )

assert resumo_cine["qtd_registros"] == resumo_cine["qtd_rotulos"], (
    "Grão violado: registros != rótulos distintos"
)
for c in colunas_cine + ["ano_censo_edicao"]:
    assert resumo_cine[f"nulos__{c}"] == 0, f"Valores nulos/vazios em {c}"

# Conferência dos rótulos com alteração de nomenclatura em 2024
display(
    df_dim_cine.filter(
        F.col("codigo_cine_rotulo").isin("0613E01", "0714E04")
    )
)

In [0]:
from pyspark.sql import functions as F

TABELA_CBO = "workspace.silver.ref_cbo_ocupacao"

# Dimensão CBO: código como STRING com 6 dígitos, descrição sem espaços nas bordas
df_dim_cbo = (
    spark.table(TABELA_CBO)
    .select(
        F.lpad(F.col("codigo").cast("STRING"), 6, "0").alias("codigo_cbo"),
        F.trim(F.col("descricao")).alias("nome_ocupacao"),
    )
)

resumo_cbo = df_dim_cbo.agg(
    F.count("*").alias("qtd_registros"),
    F.countDistinct("codigo_cbo").alias("qtd_codigos"),
    F.sum(
        F.when(
            F.col("codigo_cbo").isNull()
            | ~F.col("codigo_cbo").rlike("^[0-9]{6}$"),
            1,
        ).otherwise(0)
    ).alias("codigos_invalidos"),
    F.sum(
        F.when(
            F.col("nome_ocupacao").isNull()
            | (F.trim(F.col("nome_ocupacao")) == ""),
            1,
        ).otherwise(0)
    ).alias("descricoes_vazias"),
).collect()[0]

print("Registros:", resumo_cbo["qtd_registros"])
print("Códigos distintos:", resumo_cbo["qtd_codigos"])
print("Códigos inválidos (formato != 6 dígitos):", resumo_cbo["codigos_invalidos"])
print("Descrições nulas/vazias:", resumo_cbo["descricoes_vazias"])

if resumo_cbo["qtd_registros"] != 2778:
    print(
        f"ALERTA: esperados 2778 registros, encontrados {resumo_cbo['qtd_registros']} "
        "(dicionário pode ter sido atualizado)."
    )

assert resumo_cbo["qtd_registros"] == resumo_cbo["qtd_codigos"], (
    "Grão violado: registros != códigos distintos"
)
assert resumo_cbo["codigos_invalidos"] == 0, "Existem códigos fora do formato de 6 dígitos"
assert resumo_cbo["descricoes_vazias"] == 0, "Existem descrições nulas ou vazias"

print("\nAmostra da dimensão CBO:")
display(df_dim_cbo.orderBy("codigo_cbo").limit(10))

In [0]:
from pyspark.sql import functions as F

# Coluna identificada pelo schema real da Gold do Novo Caged: cbo2002_ocupacao (INT)
df_cbo_gold = (
    spark.table(TABELA_GOLD_CAGED)
    .select(
        F.lpad(F.col("cbo2002_ocupacao").cast("STRING"), 6, "0").alias("codigo_cbo"),
        F.col("cbo2002_ocupacao_descricao").alias("descricao_gold"),
    )
    .filter(F.col("codigo_cbo").isNotNull())
    .groupBy("codigo_cbo")
    .agg(F.max("descricao_gold").alias("descricao_gold"))
)

# CBOs presentes na Gold e ausentes na dimensão (gaps conhecidos de de-para)
df_sem_cobertura = df_cbo_gold.join(df_dim_cbo, "codigo_cbo", "left_anti")

# CBOs da dimensão não utilizados na Gold (dicionário é mais amplo — esperado)
df_nao_utilizados = df_dim_cbo.join(
    df_cbo_gold.select("codigo_cbo"), "codigo_cbo", "left_anti"
)

# Divergência de descrição para a mesma chave (Gold × dimensão)
df_divergencia_descricao = (
    df_cbo_gold
    .join(df_dim_cbo, "codigo_cbo", "inner")
    .filter(F.col("descricao_gold") != F.col("nome_ocupacao"))
)

print("CBOs distintos na Gold do Novo Caged:", df_cbo_gold.count())
print("CBOs da Gold SEM cobertura na dimensão:", df_sem_cobertura.count())
print("CBOs da dimensão não utilizados na Gold:", df_nao_utilizados.count())
print("Divergências de descrição Gold × dimensão:", df_divergencia_descricao.count())

print("\nCBOs da Gold sem cobertura na dimensão (documentados, não excluídos):")
display(df_sem_cobertura.orderBy("codigo_cbo"))

if df_divergencia_descricao.count() > 0:
    print("\nDivergências de descrição:")
    display(df_divergencia_descricao.orderBy("codigo_cbo").limit(50))

In [0]:
from pyspark.sql import functions as F


def gravar_dimensao_idempotente(
    df_dim,
    nome_tabela,
    coluna_chave,
    ddl_create,
    versao_transformacao,
):
    """Grava uma dimensão Delta de forma idempotente.

    A gravação só acontece quando:
    - a tabela ainda não existe; ou
    - o conteúdo de negócio da origem mudou (diferença simétrica); ou
    - a versão da transformação mudou.
    Do contrário, a execução é um no-op e a tabela é preservada.
    """
    # --- Validações de unicidade antes da gravação ---
    resumo = df_dim.agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct(coluna_chave).alias("qtd_chaves"),
    ).collect()[0]

    assert resumo["qtd_registros"] == resumo["qtd_chaves"], (
        f"[{nome_tabela}] Grão violado: {resumo['qtd_registros']} registros "
        f"para {resumo['qtd_chaves']} chaves distintas."
    )

    existe = spark.catalog.tableExists(nome_tabela)

    if existe:
        df_atual = spark.table(nome_tabela)

        colunas_esperadas = list(df_dim.columns) + [
            "dt_processamento",
            "versao_transformacao",
        ]
        assert list(df_atual.columns) == colunas_esperadas, (
            f"[{nome_tabela}] Schema existente diverge do esperado — "
            f"revisar manualmente antes de regravar.\n"
            f"Esperado: {colunas_esperadas}\n"
            f"Atual: {list(df_atual.columns)}"
        )

        versao_atual = df_atual.select(
            F.max("versao_transformacao")
        ).collect()[0][0]

        colunas_negocio = df_dim.columns
        mudancas = (
            df_dim.select(*colunas_negocio)
            .exceptAll(df_atual.select(*colunas_negocio))
            .count()
            + df_atual.select(*colunas_negocio)
            .exceptAll(df_dim.select(*colunas_negocio))
            .count()
        )

        if mudancas == 0 and versao_atual == versao_transformacao:
            print(
                f"[{nome_tabela}] Origem inalterada e versão "
                f"{versao_transformacao} mantida — gravação ignorada."
            )
            return False

        motivos = []
        if mudancas > 0:
            motivos.append(
                f"conteúdo da origem mudou ({mudancas} linhas diferentes)"
            )
        if versao_atual != versao_transformacao:
            motivos.append(
                f"versão da transformação {versao_atual} -> {versao_transformacao}"
            )
        print(f"[{nome_tabela}] Regravando: {'; '.join(motivos)}.")
    else:
        print(f"[{nome_tabela}] Tabela inexistente — criando pela primeira vez.")

    # --- Metadados de controle ---
    df_final = df_dim.withColumn(
        "dt_processamento", F.current_timestamp()
    ).withColumn(
        "versao_transformacao", F.lit(versao_transformacao)
    )

    if not existe:
        spark.sql(ddl_create)

    # INSERT OVERWRITE com lista explícita de colunas (alinhada ao DDL)
    lista_colunas = ", ".join(df_final.columns)
    df_final.createOrReplaceTempView("vw_dimensao_gravacao")
    spark.sql(
        f"INSERT OVERWRITE {nome_tabela} "
        f"SELECT {lista_colunas} FROM vw_dimensao_gravacao"
    )

    qtd_gravada = spark.table(nome_tabela).count()
    print(f"[{nome_tabela}] Gravação concluída: {qtd_gravada} registros.")
    return True

In [0]:
DDL_DIM_CINE = f"""
CREATE TABLE IF NOT EXISTS {DIM_CINE} (
    codigo_cine_area_geral STRING COMMENT 'Código CINE da área geral',
    nome_cine_area_geral STRING COMMENT 'Nome CINE da área geral',
    codigo_cine_area_especifica STRING COMMENT 'Código CINE da área específica',
    nome_cine_area_especifica STRING COMMENT 'Nome CINE da área específica',
    codigo_cine_area_detalhada STRING COMMENT 'Código CINE da área detalhada',
    nome_cine_area_detalhada STRING COMMENT 'Nome CINE da área detalhada',
    codigo_cine_rotulo STRING COMMENT 'Chave da dimensão: código CINE do rótulo (grão)',
    nome_cine_rotulo STRING COMMENT 'Nome CINE do rótulo (nomenclatura da edição mais recente)',
    ano_censo_edicao INT COMMENT 'Ano do Censo de onde a nomenclatura vigente foi extraída',
    dt_processamento TIMESTAMP COMMENT 'Data e hora da gravação da dimensão',
    versao_transformacao STRING COMMENT 'Versão da transformação que produziu a dimensão'
)
USING DELTA
COMMENT 'Dimensão CINE Brasil: classificação de cursos em quatro níveis, um registro por rótulo CINE'
"""

print("Tabela-alvo:", DIM_CINE)

gravou_dim_cine = gravar_dimensao_idempotente(
    df_dim=df_dim_cine,
    nome_tabela=DIM_CINE,
    coluna_chave="codigo_cine_rotulo",
    ddl_create=DDL_DIM_CINE,
    versao_transformacao=VERSAO_TRANSFORMACAO_DIMENSOES,
)

In [0]:
DDL_DIM_CBO = f"""
CREATE TABLE IF NOT EXISTS {DIM_CBO} (
    codigo_cbo STRING COMMENT 'Chave da dimensão: código CBO 2002 com 6 dígitos (zero à esquerda)',
    nome_ocupacao STRING COMMENT 'Descrição da ocupação no dicionário CBO 2002',
    dt_processamento TIMESTAMP COMMENT 'Data e hora da gravação da dimensão',
    versao_transformacao STRING COMMENT 'Versão da transformação que produziu a dimensão'
)
USING DELTA
COMMENT 'Dimensão CBO Ocupação: dicionário CBO 2002, um registro por código de ocupação'
"""

print("Tabela-alvo:", DIM_CBO)

gravou_dim_cbo = gravar_dimensao_idempotente(
    df_dim=df_dim_cbo,
    nome_tabela=DIM_CBO,
    coluna_chave="codigo_cbo",
    ddl_create=DDL_DIM_CBO,
    versao_transformacao=VERSAO_TRANSFORMACAO_DIMENSOES,
)

In [0]:
from pyspark.sql import functions as F

dimensoes = [
    (DIM_CINE, "codigo_cine_rotulo", 372),
    (DIM_CBO, "codigo_cbo", 2778),
]

for nome_tabela, chave, esperado in dimensoes:
    print(f"\n== {nome_tabela} ==")

    df_gold = spark.table(nome_tabela)

    resumo = df_gold.agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct(chave).alias("qtd_chaves"),
        F.max("versao_transformacao").alias("versao"),
        F.max("dt_processamento").alias("dt_ultimo_processamento"),
    ).collect()[0]

    print("Registros persistidos:", resumo["qtd_registros"])
    print("Chaves distintas:", resumo["qtd_chaves"])
    print("Versão da transformação:", resumo["versao"])
    print("Último processamento:", resumo["dt_ultimo_processamento"])

    assert resumo["qtd_registros"] == resumo["qtd_chaves"], (
        f"[{nome_tabela}] Unicidade violada na tabela persistida"
    )

    if resumo["qtd_registros"] != esperado:
        print(
            f"ALERTA: contagem {resumo['qtd_registros']} difere do esperado {esperado} "
            "(base pode ter sido atualizada)."
        )

    display(df_gold.orderBy(chave).limit(10))

## Conclusões

* **`workspace.gold.dim_cine_brasil`** — 372 registros, um por `codigo_cine_rotulo`, com os quatro níveis CINE preservados e a nomenclatura da edição mais recente (`ano_censo_edicao` registra a origem). Zero conflitos no mesmo ano; `0613E01` = Engenharia de software e `0714E04` = Engenharia de computação (ambos da edição 2024).
* **`workspace.gold.dim_cbo_ocupacao`** — 2.778 registros, código CBO 2002 como STRING de 6 dígitos; zero duplicidades, códigos inválidos ou descrições vazias.
* **Cobertura CBO na Gold do Novo Caged** — 2.727 códigos distintos na `fato_caged_semestral`; **41 sem cobertura** na dimensão (todos com descrição nula na Gold — gaps de de-para do dicionário, documentados e não excluídos); 92 códigos da dimensão não utilizados na Gold (dicionário mais amplo, esperado); **0 divergências de descrição** entre Gold e dimensão.
* **Gravação idempotente validada** — reexecução com origem e versão inalteradas ignora a gravação e preserva `dt_processamento` original.
* **Pendente (próxima etapa):** `bridge_cine_cbo` — a Golds existentes não foram alteradas.